REDROBS SUBMISSION

# REDROBS Submission

## Important Dependencies to Run the Pipeline

**Note:** While running the pipeline, you may encounter environment or version compatibility issues. We have listed the compatible dependencies required to ensure smooth execution.

It is **strongly recommended** to restart the runtime/session after completing each notebook cell or major execution stage to ensure a stable and seamless experience.

---

## Shared Google Drive (Recommended)

We kindly request the host team to use our shared Google Drive folder, as it contains all the compatible resources required for execution, including:

* Pre-trained models
* Datasets
* Required assets
* Final submission files

Please add the shared folder to **"My Drive"** on your system before running the pipeline.

**Google Drive Link:**

https://drive.google.com/drive/folders/142NtL5JBf89QrY2ZMPu6RNVET7wAwKkf?usp=drive_link

---
## Important Note

Any submission files (including CSV files) present within this folder are **internal experimental or dummy files** created by the team during development and testing. **They are not the official submission files.**

Kindly **do not consider any CSV file or intermediate output present in this folder as part of our final submission.** Only the files explicitly designated as the final submission should be evaluated.


## Pipeline Execution Modes

This pipeline is optimized to work in **two modes**:

* **Production Mode**
* **Testing Mode**

During development, due to time constraints and the temporary nature of Google Colab sessions, we configured the production Qdrant storage to use the local **`/content`** directory by default instead of Google Drive.

If you wish to preserve the generated Qdrant database across sessions, you may change the storage path to a folder in Google Drive. However, this is **not necessary** for a single runtime execution and will not cause any issues during one-time evaluation.

### Why did we choose this approach?

Writing directly to Google Drive from Google Colab is significantly slower than writing to the local `/content` directory. To optimize execution speed, our team allowed the pipeline to operate entirely within the local storage during runtime. Once execution was complete, we manually downloaded the generated Qdrant database and moved it to Google Drive for permanent storage.

We will also include these pre-generated files in the GitHub repository for reference and convenience.


PLZ INSTALL FOLLOWIND DEPENDENCIES BEFORE MOVING AHEAD

In [1]:
!pip install -qU transformers vllm qdrant-client sentence-transformers json-repair python-docx pandas numpy
!pip install "transformers==4.47.1" --upgrade
!pip install torch==2.4.0 vllm==0.6.3 xformers==0.0.27.post2 optimum[onnxruntime] python-docx qdrant-client FlagEmbedding
!pip install "diffusers<0.35.0"
!pip uninstall -y torchaudio torchvision # Prevents vLLM compatibility crashes in Colab
!pip uninstall -y torchcodec
!pip install torch==2.4.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 279.2/279.2 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.0/14.0 MB 102.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.8/178.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 61.5 MB/s eta 0:00:00
  Attempting uninstall: diffusers
    Found existing installation: diffusers 0.38.0
    Uninstalling diffusers-0.38.0:
      Successfully uninstalled diffusers-0.38.0
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.19.0
Uninstalling torchvision-0.19.0:
  Successfully uninstalled torchvision-0.19.0
Found existing installation: torchcodec 0.11.0+cu128
Uninstalling torchcodec-0.11.0+cu128:
  Successfully uninstalled torchcodec-0.11.0+cu128


CONNECT DRIVE AND SHARED FOLDER

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


NOT WE HAVE OPTIMIZED PIPELINE TO WORK BOTH IN PRODUCTION VS TEST PHASE DO TOGGLE IT ACCORDING TO YOUR WISH AS TRUE/FALSE


ALSO WE WOULD RECCOMEND TO CHANGE OR EDIT THE QDRANT FOLDER DEFAULT PATH TO AVOID AVOID QDRANT LOCK ERROR FOR EXAMPLE HERE BY DEFAULT IS "TEST_v58_qdrant" plz change it to other name before using

PHASE 1

In [36]:
import os, sys, gc, json, ast, math, hashlib

# 🛑 CRITICAL FIX FOR COLAB
if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: 1
if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2
os.environ["VLLM_USE_V1"] = "0"

import pandas as pd
import numpy as np
import torch
from datetime import datetime
from tqdm import tqdm
from qdrant_client import QdrantClient
from qdrant_client.models import HnswConfigDiff
from qdrant_client.models import (
    PointStruct, VectorParams, Distance, PointIdsList
)
from huggingface_hub import snapshot_download
import json_repair

try:
    from dateutil import parser as _dp
    def _parse_date(s):
        try: return _dp.parse(str(s)) if s and not pd.isna(s) else None
        except: return None
except ImportError:
    def _parse_date(s): return None

from vllm import LLM, SamplingParams
from sentence_transformers import SentenceTransformer

# ══════════════════════════════════════════════════════════
# ⚙️ CONFIGURATION & TEST TOGGLE
# ══════════════════════════════════════════════════════════
IS_TEST_RUN = True
TEST_LIMIT  = 200

DRIVE_DIR          = "/content/drive/MyDrive/redrob_ai"
DATA_PATH          = os.path.join(DRIVE_DIR, "candidates_full_dataset.csv")
JD_PATH            = os.path.join(DRIVE_DIR, "job_description.docx")

MODELS_DIR         = os.path.join(DRIVE_DIR, "models")
QWEN_7B_PATH       = os.path.join(MODELS_DIR, "Qwen2.5-7B-Instruct-AWQ")
QWEN_3B_PATH       = os.path.join(MODELS_DIR, "Qwen2.5-3B-Instruct-AWQ")
BGE_M3_PATH        = os.path.join(MODELS_DIR, "bge-m3")

if IS_TEST_RUN:
    print(f"⚠️ RUNNING IN TEST MODE ({TEST_LIMIT} Candidates)")
    PHASE1A_CSV  = os.path.join(DRIVE_DIR, "TEST_v13_phase1a.csv")
    PHASE1B_CSV  = os.path.join(DRIVE_DIR, "TEST_v8.2.2_phase1b.csv")
    QDRANT_PATH  = os.path.join(DRIVE_DIR, "TEST_v58_qdrant")
    TARGET_ROWS  = TEST_LIMIT
else:
    print("🔥 RUNNING IN PRODUCTION MODE (100,000 Candidates)")
    PHASE1A_CSV  = os.path.join(DRIVE_DIR, "prod_v004_phase1a.csv")
    PHASE1B_CSV  = os.path.join(DRIVE_DIR, "prod_v004_phase1b.csv")
    QDRANT_PATH  = "/content/qdrant_storage"
    TARGET_ROWS  = 100_000

CHUNK_SIZE     = 2_500
TOP_QDRANT_N   = 300
RERANK_N       = 150
SUBMISSION_N   = 100
HP_THRESHOLD   = 0.80

QUANT_3B = "awq"
QUANT_7B = "awq"

# ══════════════════════════════════════════════════════════
# SCHEMA & KEYWORDS
# ══════════════════════════════════════════════════════════
JD_CORE_SKILLS = {
    "embedding", "vector database", "hybrid search", "qdrant", "faiss", "milvus",
    "retrieval", "reranker", "bm25", "semantic search", "rag", "fine-tuning",
    "evaluation", "ndcg", "mrr", "map", "python", "production ml", "ranking",
    "information retrieval", "dense retrieval", "sparse retrieval", "sentence transformers",
    "cross-encoder", "bi-encoder", "ann", "hnsw", "pgvector",
}

DEPLOYMENT_KEYWORDS = {
    "served", "deployed", "production", "api", "inference", "throughput",
    "latency", "pipeline", "a/b test", "scaled", "realtime", "sla", "qps",
    "release", "shipped", "live", "requests per second", "endpoint",
    "vercel", "render", "railway", "netlify", "supabase", "ci/cd",
    "webhooks", "serverless", "cloud", "hosting", "containerized",
    "docker", "infrastructure", "latency-optimized", "websockets",
    "environment", "provisioned", "uptime", "build", "deployment-failure",
}

FRAMEWORK_WRAPPER_TERMS = {
    "langchain", "crewai", "autogpt", "dspy", "llamaindex",
    "flowise", "langflow", "agentgpt", "babyagi",
    "langgraph", "haystack", "semantic-kernel", "mastra",
    "ag2", "camel", "swarm", "metagpt",
    "pydantic-ai", "smolagents", "adk",
}

EXTRACTION_SCHEMA_STRING = """{
    "ai_skills": ["array of strings"],
    "deployment_evidence": boolean,
    "eval_metrics_exp": boolean,
    "scale_evidence": boolean,
    "domain": "nlp_ir" OR "cv_robotics" OR "general_ml" OR "data_eng" OR "software_only" OR "other",
    "product_co_evidence": boolean,
    "implicit_signal": "string",
    "coding_recency_years": integer
}"""

ASPECT_WEIGHTS = {"skills": 0.50, "career": 0.40, "education": 0.10}

# ══════════════════════════════════════════════════════════
# UTILITIES & SETUP
# ══════════════════════════════════════════════════════════
def flush():
    gc.collect()
    torch.cuda.empty_cache()
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
        destroy_model_parallel()
        destroy_distributed_environment()
    except Exception:
        pass
    gc.collect()
    torch.cuda.empty_cache()

def setup_local_models():
    print("=" * 55 + "\n📥 CHECKING LOCAL MODEL STORAGE\n" + "=" * 55)
    os.makedirs(MODELS_DIR, exist_ok=True)
    models_to_check = {
        "Qwen/Qwen2.5-7B-Instruct-AWQ": QWEN_7B_PATH,
        "Qwen/Qwen2.5-3B-Instruct-AWQ": QWEN_3B_PATH,
        "BAAI/bge-m3": BGE_M3_PATH,
    }
    for repo_id, local_path in models_to_check.items():
        if not os.path.exists(local_path) or not os.listdir(local_path):
            snapshot_download(repo_id=repo_id, local_dir=local_path)
    print("✅ Models ready.\n")

def safe_eval(val):
    if pd.isna(val) if not isinstance(val, (dict, list)) else False: return {}
    if isinstance(val, (dict, list)): return val
    try: return ast.literal_eval(str(val).strip())
    except: return {}

def safe_list(val):
    r = safe_eval(val)
    return r if isinstance(r, list) else []

def cid_to_point_id(cid) -> int:
    return int(hashlib.md5(str(cid).encode()).hexdigest()[:15], 16) % (2 ** 53)

def clean_json_output(raw_text: str) -> str:
    raw = raw_text.strip()
    if raw.startswith("```json"): raw = raw[7:]
    if raw.startswith("```"):     raw = raw[3:]
    if raw.endswith("```"):       raw = raw[:-3]
    return raw.strip()

# ══════════════════════════════════════════════════════════
# INTELLIGENCE ENGINE
# ══════════════════════════════════════════════════════════
def compute_honeypot_risk(career: list, profile: dict, skills: list) -> tuple:
    risk  = 0.0
    flags = []
    tenure_months = 0
    spans = []
    for r in career:
        if not isinstance(r, dict): continue
        s = _parse_date(r.get("start_date"))
        e = _parse_date(r.get("end_date")) or datetime.now()
        if s and e > s:
            tenure_months += (e.year - s.year) * 12 + (e.month - s.month)
            spans.append((s, e))

    claimed_months = float(profile.get("years_of_experience", 0) or 0) * 12
    if claimed_months > 0 and abs(tenure_months - claimed_months) > 30:
        risk += 0.40; flags.append("tenure_yoe_mismatch")

    if len(spans) >= 2:
        cal_months = ((max(e for _, e in spans) - min(s for s, _ in spans)).days / 30) or 1
        if tenure_months / cal_months > 1.15:
            risk += 0.30; flags.append("concurrent_role_overlap")

    for sk in skills:
        if not isinstance(sk, dict): continue
        prof = str(sk.get("proficiency", "")).lower()
        asmt = float(sk.get("assessment_score", 50) or 50)
        if prof in ("beginner", "novice") and asmt > 85:
            risk += 0.10; flags.append("assessment_proficiency_mismatch"); break
        if prof == "expert" and asmt < 25:
            risk += 0.20; flags.append("assessment_proficiency_mismatch"); break

    return min(risk, 1.0), flags

def compute_behavioral_composite(sigs: dict, profile: dict) -> tuple:
    penalties = 0.0
    boosts    = 0.0
    flags     = []

    notice = int(sigs.get("notice_period_days", 60) or 60)
    notice_m = (1.00 if notice <= 30 else 0.97 if notice <= 60
                else 0.95 if notice <= 90 else 0.90 if notice <= 120 else 0.55)

    last_active = _parse_date(sigs.get("last_active_date") or sigs.get("last_active"))
    if last_active:
        inactive = (datetime.now() - last_active).days
        if   inactive > 540: penalties += 0.45; flags.append("inactive_18m+")
        elif inactive > 180: penalties += 0.32; flags.append("inactive_6m+")
        elif inactive > 90:  penalties += 0.5

    resp = float(sigs.get("recruiter_response_rate", -1) or -1)
    if   resp > 0.70:       boosts    += 0.08
    elif 0 < resp < 0.20:   penalties += 0.05; flags.append("low_response_rate")

    accept = float(sigs.get("offer_acceptance_rate", -1) or -1)
    if   accept > 0.80:     boosts    += 0.03
    elif 0 < accept < 0.20: penalties += 0.02; flags.append("low_offer_acceptance")

    saved = int(sigs.get("saved_by_recruiters_30d", 0) or 0)
    if   saved >= 5: boosts += 0.08
    elif saved >= 2: boosts += 0.03

    github = float(sigs.get("github_activity_score", -1) or -1)
    g_norm = (github / 10.0) if github >= 0 else 0.0
    if   g_norm > 0.70:                     boosts    += 0.15
    elif github < 0 or g_norm < 0.10:       penalties += 0.03; flags.append("no_public_code")

    li_conn      = int(sigs.get("linkedin_connections", 0) or 0)
    endorsements = int(sigs.get("endorsements_count",   0) or 0)
    ext_val = (0.6 * g_norm
             + 0.3 * min(li_conn / 500, 1.0)
             + 0.1 * min(endorsements / 50, 1.0))
    if ext_val < 0.10: penalties += 0.025

    sal_str = str(sigs.get("expected_salary_range_inr_lpa", "") or "")
    try:
        nums = [float(x) for x in sal_str.replace(" ", "").split("-")
                if x.replace(".", "").isdigit()]
        if nums and max(nums) > 120:
            penalties += 0.08; flags.append("salary_mismatch")
    except Exception:
        pass

    completeness = float(sigs.get("profile_completeness", 0.5) or 0.5)
    if completeness < 0.4: penalties += 0.03

    loc      = str(profile.get("location", sigs.get("location", "")) or "").lower()
    relocate = str(sigs.get("willing_to_relocate", "false")).lower() in ("true", "yes", "1")
    mode     = str(sigs.get("preferred_work_mode", "") or "").lower()

    if   any(c in loc for c in ["pune", "noida"]):
        loc_m = 1.00
    elif any(c in loc for c in ["hyderabad", "mumbai", "delhi", "bengaluru", "bangalore", "ncr"]):
        loc_m = 0.95
    elif "india" in loc:
        loc_m = 0.95 if relocate else 0.92
    elif relocate:
        loc_m = 0.90
    else:
        loc_m = 0.90; flags.append("outside_india_no_relocation")

    if "remote" in mode and "hybrid" not in mode:
        loc_m = min(loc_m, 0.90)

    behavioral_score = min(max(0.50 + boosts - penalties, 0.05), 1.0)
    return behavioral_score, loc_m, notice_m, flags, notice

def compute_rule_signals(career: list) -> tuple:
    penalties = 0.0
    boosts    = 0.0
    flags     = []

    if not career: return penalties, boosts, flags

    desc = " ".join(str(r.get("description", "") if isinstance(r, dict) else "").lower()
                    for r in career)
    role_titles = [str(r.get("title", "") if isinstance(r, dict) else "").lower()
                   for r in career]

    has_wrapper = any(t in desc for t in FRAMEWORK_WRAPPER_TERMS)
    has_pre_llm = any(str(y) in desc for y in range(2010, 2022))
    if has_wrapper and not has_pre_llm:
        penalties += 0.15; flags.append("shallow_ai_era")
    elif any(str(y) in desc for y in range(2015, 2020)) and "nlp" in desc:
        boosts += 0.08

    research_role_count = sum(
        1 for t in role_titles
        if any(term in t for term in {"researcher", "scientist", "postdoc", "fellow", "phd intern"})
    )
    if len(career) > 0 and (research_role_count / len(career)) > 0.70:
        penalties += 0.35; flags.append("research_only")

    if not any(kw in desc for kw in DEPLOYMENT_KEYWORDS) and len(career) >= 4:
        penalties += 0.05; flags.append("no_deployment_evidence")

    if role_titles:
        latest = role_titles[-1]
        if any(kw in latest for kw in ["director", "vp ", "head of", "chief"]) and "engineer" not in latest:
            penalties += 0.08; flags.append("mgmt_only_recent")

    return penalties, boosts, flags

def score_from_extraction(raw_json: str) -> tuple:
    flags = []
    try:
        ext = json_repair.loads(raw_json)
        if not isinstance(ext, dict): ext = {}
    except Exception:
        return 0.15, ["extraction_parse_failed"]

    score     = 0.0
    ai_skills = {s.lower() for s in ext.get("ai_skills", [])}
    overlap   = len(ai_skills & JD_CORE_SKILLS)
    score += min(overlap / max(len(JD_CORE_SKILLS) * 0.25, 1), 1.0) * 0.40

    if ext.get("deployment_evidence"): score += 0.20


    if ext.get("eval_metrics_exp"): score += 0.10
    if ext.get("scale_evidence"):   score += 0.05

    domain_map = {
        "nlp_ir": 1.0, "general_ml": 0.25, "data_eng": 0.30,
        "software_only": 0.10, "cv_robotics": 0.10, "other": 0.08,
    }
    score += domain_map.get(ext.get("domain", "other"), 0.10) * 0.30

    if ext.get("product_co_evidence"): score += 0.10

    recency = int(ext.get("coding_recency_years", 0) or 0)
    if   recency > 18: score += 0.10; flags.append(f"inactive_{recency}yr_coding")
    elif recency > 12: score -= 0.08

    return max(0.0, min(score, 1.0)), flags

# ══════════════════════════════════════════════════════════
# PHASE 0: JD INTELLIGENCE
# ══════════════════════════════════════════════════════════
def run_phase0_jd_parsing(jd_text: str) -> dict:
    print("\n" + "=" * 55 + "\n📜 PHASE 0: JD INTELLIGENCE\n" + "=" * 55)

    jd_llm = LLM(
        model=QWEN_7B_PATH,
        quantization=QUANT_7B,
        max_model_len=4096,
        gpu_memory_utilization=0.70,
        swap_space=1,
        max_num_seqs=1,
        dtype="float16",
    )
    params = SamplingParams(temperature=0.0, max_tokens=1024)
    SYS = "You are a precise technical recruiter,you will be analyzing job description. Return valid JSON only, no markdown, no explanation."

    pass_prompts = {
        "explicit": (
            "Extract the explicit technical requirements from this job description as JSON. "
            "Keys: required_skills (array), required_experience_years (int), "
            "must_have_deployment_experience (bool), must_have_eval_framework_exp (bool), "
            "preferred_domain (string).\n\nJD:\n" + jd_text[:3000]
        ),
        "implicit": (
            "What does this JD IMPLICITLY require that is not stated directly? "
            "Return JSON with: implicit_skills (array), cultural_requirements (array), "
            "implicit_signals (array of brief strings describing unstated expectations).\n\nJD:\n"
            + jd_text[:3000]
        ),
        "anti": (
            "List all DISQUALIFYING backgrounds this JD mentions explicitly or implicitly. "
            "Return JSON with: disqualified_backgrounds (array), red_flag_titles (array), "
            "red_flag_company_types (array).\n\nJD:\n" + jd_text[:3000]
        ),
    }

    results = {}
    for pass_name, user_prompt in pass_prompts.items():
        prompt = (f"<|im_start|>system\n{SYS}\n<|im_end|>\n"
                  f"<|im_start|>user\n{user_prompt}\n<|im_end|>\n"
                  f"<|im_start|>assistant\n{{")
        raw = jd_llm.generate([prompt], params)[0].outputs[0].text.strip()
        raw = clean_json_output("{" + raw)
        try: results[pass_name] = json.loads(raw)
        except Exception: results[pass_name] = {"raw": raw}
        print(f"  ✓ {pass_name} pass")

    del jd_llm; flush()

    all_kw = set(JD_CORE_SKILLS)
    if isinstance(results.get("explicit"), dict):
        all_kw.update(s.lower() for s in results["explicit"].get("required_skills", []))
    if isinstance(results.get("implicit"), dict):
        all_kw.update(s.lower() for s in results["implicit"].get("implicit_skills", []))

    jd_context = {
        "explicit":            results.get("explicit", {}),
        "implicit":            results.get("implicit", {}),
        "anti":                results.get("anti", {}),
        "keywords_for_prompt": ", ".join(sorted(all_kw)[:30]),
        "full_jd_snippet":     jd_text[:1500],
    }
    print("✅ Phase 0 complete — jd_context ready.\n")
    return jd_context

# ══════════════════════════════════════════════════════════
# PHASE 1A: MASS EXTRACTION
# ══════════════════════════════════════════════════════════
def run_phase1a_mass_extraction(csv_input: str, jd_context: dict):
    print("=" * 55 + "\n⚡ PHASE 1A: MASS EXTRACTION\n" + "=" * 55)

    processed_ids  = set()
    header_written = False

    if os.path.exists(PHASE1A_CSV):
        try:
            existing_df   = pd.read_csv(PHASE1A_CSV, usecols=["candidate_id"])
            processed_ids = set(existing_df["candidate_id"].tolist())
            header_written = True
            print(f"🔄 Checkpoint Found: {len(processed_ids)} candidates already processed.")
        except Exception:
            os.remove(PHASE1A_CSV)

    if len(processed_ids) >= TARGET_ROWS:
        return

    extractor = LLM(
        model=QWEN_7B_PATH,
        quantization=QUANT_3B,
        max_model_len=3072,
        gpu_memory_utilization=0.75,
        swap_space=1,
        max_num_seqs=64,
        enable_prefix_caching=True,
        dtype="float16",
    )

    params = SamplingParams(temperature=0.0, max_tokens=300)
    jd_kw  = jd_context.get("keywords_for_prompt", "embeddings, retrieval, production, ranking")
    SYSTEM = (
        "You are a strict, deterministic data extraction engine. Your sole function is to map explicit, professional engineering facts from the candidate's profile into a predefined JSON structure.\n\n"
        "CRITICAL GUARDRAILS:\n"
        "1. ZERO INFERENCE: Do not assume, guess, or deduce any capabilities. If a skill or metric is not explicitly written in the text, it does not exist.\n"
        "2. NO HALLUCINATION: Extract only real, established technical AI/ML engineering skills directly mentioned. Do not invent frameworks.\n"
        "3. STRICT BOOLEANS: Set boolean fields to `true` ONLY if concrete, explicit professional evidence exists. If unsure, default to `false`.\n"
        "4. MISSING DATA: If information for a field is absent, output `[]`, `false`, `null`, or `0` depending on the schema type.\n\n"
        "HOBBYIST & LEARNER FILTERS (STRICT EXCLUSIONS):\n"
        "- IGNORE CONSUMER LLM USAGE: If a candidate mentions 'using ChatGPT', 'playing with Midjourney', 'prompt engineering', or generic AI usage, DO NOT extract these as AI skills. We are looking for engineers who *build* models, not consumers who talk to them.\n"
        "-IGNORE LEARNERS WHO ARE DOING ANY COURSE OR STUFF WE WANT PRACTICAL TALENT"
        "- BE AWARE OF GENRIC AI TERMINOLOGY WRITTEN BY CANDIDATE IGNORE IT IF NOT PROFFESIONALLY DEVLOPMENT CONCENTRATED"
        "- IGNORE LEARNER SIGNALS: If a candidate is merely 'learning AI', taking a basic tutorial, or describes themselves as an 'AI enthusiast', treat them as having NO professional AI skills. Their `ai_skills` must be left empty `[]`.\n\n"
        "FIELD DEFINITIONS:\n"
        "- ai_skills: [List exact engineering tools, architectures, algorithms, or vector DBs built/implemented (e.g., PyTorch, Qdrant, RAG, CNNs). Completely omit generic terms like 'AI', 'LLM', or 'ChatGPT'.]\n"
        "- ai skills are proffesional ai devlopment related domain not using ai should be listed.Realistic ai skills are real ai domain related stuff\n"
        "- deployment_evidence: [true ONLY if the text contains professional verbs showing they served, productionized, or shipped code to users.]\n"
        "- domain: [Classify as 'nlp_ir', 'cv_robotics', or 'general_ml' ONLY if they have concrete engineering skills. If they are a standard developer just 'learning' or using ChatGPT, you MUST classify them as 'software_only' or 'other'.]\n"
        "- coding_recency_years: [Calculate strictly from the most recent technical role provided.]\n\n"
        f"Target JSON Schema:\n{EXTRACTION_SCHEMA_STRING}\n\n"
        "OUTPUT REQUIREMENT:\n"
        "Output absolutely nothing except a single, valid JSON object. Do not include markdown blocks, explanations, or introductory text."
    )
    num_chunks = math.ceil(TARGET_ROWS / CHUNK_SIZE)

    for chunk_idx, chunk_df in enumerate(
        tqdm(pd.read_csv(csv_input, chunksize=CHUNK_SIZE, nrows=TARGET_ROWS), total=num_chunks, desc="1A chunks")
    ):
        chunk_df = chunk_df[~chunk_df["candidate_id"].isin(processed_ids)]
        if chunk_df.empty: continue

        blobs, prompts = [], []

        for _, row in chunk_df.iterrows():
            prof   = safe_eval(row.get("profile",        "{}"))
            career = safe_list(row.get("career_history", "[]"))
            skills = safe_list(row.get("skills",         "[]"))
            sigs   = safe_eval(row.get("redrob_signals", "{}"))

            desc_text   = " ".join(str(r.get("description", "") if isinstance(r, dict) else "").lower() for r in career)
            blob_skills = " ".join(s.get("name", "") for s in skills if isinstance(s, dict))
            blob_career = f"{prof.get('summary', '')} {desc_text}"[:1200]
            blob_edu    = str(safe_eval(row.get("education", "{}")))[:400]

            hp_risk, hp_flags                          = compute_honeypot_risk(career, prof, skills)
            beh_score, loc_m, notice_m, beh_flags, notice_days = compute_behavioral_composite(sigs, prof)
            rule_pen, rule_boost, rule_flags           = compute_rule_signals(career)
            rule_score = min(max(0.5 + rule_boost - rule_pen, 0.05), 1.0)

            all_flags = beh_flags + rule_flags + hp_flags

            blobs.append({
                "candidate_id":      row["candidate_id"],
                "point_id":          cid_to_point_id(row["candidate_id"]),
                "blob_skills":       blob_skills,
                "blob_career":       blob_career,
                "blob_education":    blob_edu,
                "behavioral_score":  round(beh_score,  4),
                "rule_score":        round(rule_score,  4),
                "loc_multiplier":    round(loc_m,       4),
                "notice_multiplier": round(notice_m,    4),
                "honeypot_risk":     round(hp_risk,     4),
                "notice_days":       notice_days,
                "red_flags":         " | ".join(all_flags) if all_flags else "",
                "composite_score":   0.0,
                "extraction_score":  0.0,
                "extracted_json":    "",
            })

            prompts.append(
                f"<|im_start|>system\n{SYSTEM}\n<|im_end|>\n"
                f"<|im_start|>user\nProfile:\n{blob_career}\n<|im_end|>\n"
                f"<|im_start|>assistant\n{{"
            )

        outputs = extractor.generate(prompts, params)

        for i, out in enumerate(outputs):
            raw_json  = clean_json_output("{" + out.outputs[0].text.strip())
            ext_score, ext_flags = score_from_extraction(raw_json)

            blobs[i]["extraction_score"] = round(ext_score, 4)
            blobs[i]["extracted_json"]   = raw_json
            if ext_flags:
                blobs[i]["red_flags"] = (blobs[i]["red_flags"] + " | " + " | ".join(ext_flags)).strip(" | ")

            blobs[i]["composite_score"] = round(
                (0.40 * ext_score + 0.35 * blobs[i]["rule_score"] + 0.25 * blobs[i]["behavioral_score"])
                * blobs[i]["loc_multiplier"] * blobs[i]["notice_multiplier"], 4
            )

        df_out = pd.DataFrame(blobs)
        df_out.to_csv(PHASE1A_CSV, mode="a", header=(not header_written), index=False)
        header_written = True

        del chunk_df, blobs, prompts, outputs, df_out
        gc.collect()

    del extractor; flush()
    print(f"✅ Phase 1A complete → {PHASE1A_CSV}\n")

# ══════════════════════════════════════════════════════════
# PHASE 1B: EVIDENCE REASONING
# ══════════════════════════════════════════════════════════
def build_multistrat_top1k(df: pd.DataFrame) -> pd.DataFrame:
    def safe_nlargest(df_sub, n, col):
        if col not in df_sub.columns or df_sub.empty:
            return pd.DataFrame(columns=df.columns)
        return df_sub.nlargest(n, col)

    seen = set()
    s1 = safe_nlargest(df, 550, "composite_score")
    seen.update(s1["candidate_id"].tolist())

    s2 = safe_nlargest(df[~df["candidate_id"].isin(seen)], 150, "behavioral_score")
    seen.update(s2["candidate_id"].tolist())

    def has_deployment_lang(text: str) -> bool:
        t = str(text).lower()
        return sum(kw in t for kw in DEPLOYMENT_KEYWORDS) >= 3

    remaining_3 = df[~df["candidate_id"].isin(seen)]
    if "blob_career" in remaining_3.columns:
        mask = remaining_3["blob_career"].apply(has_deployment_lang)
        s3   = safe_nlargest(remaining_3[mask], 150, "composite_score")
        seen.update(s3["candidate_id"].tolist())
    else:
        s3 = pd.DataFrame(columns=df.columns)

    remaining_4 = df[~df["candidate_id"].isin(seen)]
    if "composite_score" in remaining_4.columns:
        mid        = remaining_4["composite_score"].between(0.35, 0.65)
        notice_col = remaining_4.get("notice_days", pd.Series(60, index=remaining_4.index))
        s4         = safe_nlargest(remaining_4[mid & (notice_col <= 30)], 100, "composite_score")
    else:
        s4 = pd.DataFrame(columns=df.columns)

    combined = pd.concat([s1, s2, s3, s4]).drop_duplicates(subset="candidate_id").head(1000)
    print(f"  Strata pool selected: {len(combined)} candidates")
    return combined

def run_phase1b_evidence_reasoning(jd_context: dict) -> pd.DataFrame:
    print("=" * 55 + "\n🧠 PHASE 1B: EVIDENCE REASONING\n" + "=" * 55)

    df_1a  = pd.read_csv(PHASE1A_CSV)
    df_top = build_multistrat_top1k(df_1a)
    del df_1a; gc.collect()

    reasoner = LLM(
        model=QWEN_7B_PATH,
        quantization=QUANT_7B,
        max_model_len=4096,
        gpu_memory_utilization=0.75,
        swap_space=1,
        max_num_seqs=16,
        enable_prefix_caching=True,
        dtype="float16",
    )
    params = SamplingParams(temperature=0.1, max_tokens=450)

    anti     = jd_context.get("anti", {})
    implicit = jd_context.get("implicit", {})

    REASONING_SYS = (
        "You are a senior technical recruiter screening for a retrieval/search ML engineer role.\n"
        f"Job context: {jd_context.get('full_jd_snippet', '')[:600]}\n\n"
        f"Disqualifiers: {json.dumps(anti.get('disqualified_backgrounds', []))}\n"
        f"Implicit signals: {json.dumps(implicit.get('implicit_signals', []))}\n\n"
        "Task: find IMPLICIT evidence of fit that keyword matching misses.\n"
        "Also check: does the career description contain CONCRETE deployment evidence be very optimistic "
        "(words or semantically similar words like: served, shipped, production, endpoint, latency, QPS, A/B test, pipeline) or semainticaly similar words which depicts deployement skills of candidate?\n"
        "Return pure JSON — no markdown, no explanation:\n"
        '{"implicit_ai_evidence": "<string>", '
        '"has_deployment_evidence": true|false, '
        '"evidence_strength": "strong|moderate|weak|none", '
        '"key_concern": "<string or null>", '
        '"top_skill_signal": "<single most relevant skill or phrase from career text>"}'
    )

    prompts = [
        f"<|im_start|>system\n{REASONING_SYS}\n<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Skills blob: {str(row.get('blob_skills', ''))[:400]}\n"
        f"Career:\n{str(row.get('blob_career', ''))[:1000]}\n"
        f"<|im_end|>\n<|im_start|>assistant\n{{"
        for _, row in df_top.iterrows()
    ]

    outputs   = reasoner.generate(prompts, params)
    enriched  = []
    boost_map = {"strong": 0.20, "moderate": 0.10, "weak": 0.03, "none": 0.0}

    for i, out in enumerate(outputs):
        raw = clean_json_output("{" + out.outputs[0].text.strip())
        try:
            parsed = json_repair.loads(raw)
            if not isinstance(parsed, dict): parsed = {}
        except Exception:
            parsed = {}

        strength   = parsed.get("evidence_strength", "weak")
        boost      = boost_map.get(strength, 0.0)
        base_score = float(df_top.iloc[i].get("composite_score", 0.0))
        hp_risk    = float(df_top.iloc[i].get("honeypot_risk",   0.0))

        enriched_score = 0.10 if hp_risk >= 0.80 else min(base_score + boost, 1.0)
        deployment_found = bool(parsed.get("has_deployment_evidence", False))

        enriched.append({
            "implicit_evidence":        parsed.get("implicit_ai_evidence", ""),
            "evidence_strength":        strength,
            "key_concern":              parsed.get("key_concern"),
            "top_skill_signal":         parsed.get("top_skill_signal", ""),
            "deployment_confirmed_1b":  deployment_found,
            "enriched_composite_score": round(enriched_score, 4),
        })

    df_top = df_top.reset_index(drop=True)
    new_cols = pd.DataFrame(enriched)
    for col in new_cols.columns:
        df_top[col] = new_cols[col]

    df_top.to_csv(PHASE1B_CSV, index=False)
    del reasoner; flush()
    print(f"✅ Phase 1B complete → {PHASE1B_CSV}\n")
    return df_top

# ══════════════════════════════════════════════════════════
# PHASE 1C: EMBEDDING + QDRANT INGEST
# ══════════════════════════════════════════════════════════
def run_phase1c_full_vector_ingest() -> QdrantClient:
    print("=" * 55 + "\n🧬 PHASE 1C: BGE-M3 ENCODING\n" + "=" * 55)
    client = QdrantClient(path=QDRANT_PATH)

    for aspect in ASPECT_WEIGHTS:
        try: client.delete_collection(f"candidates_{aspect}")
        except Exception: pass
        client.create_collection(
            f"candidates_{aspect}",
            vectors_config=VectorParams(
                size=1024,
                distance=Distance.COSINE,
                on_disk=True
            ),
            hnsw_config=HnswConfigDiff(
                on_disk=True
            )
        )

    bge = SentenceTransformer(BGE_M3_PATH)
    bge.max_seq_length = 512
    num_chunks = math.ceil(TARGET_ROWS / CHUNK_SIZE)

    for _, chunk_df in enumerate(tqdm(pd.read_csv(PHASE1A_CSV, chunksize=CHUNK_SIZE), total=num_chunks)):
        for aspect in ASPECT_WEIGHTS:
            col   = f"blob_{aspect}"
            texts = chunk_df[col].astype(str).fillna("").tolist() if col in chunk_df.columns else [""] * len(chunk_df)
            vecs  = bge.encode(texts, batch_size=128)

            points = [
                PointStruct(
                    id=int(row["point_id"]),
                    vector=vecs[j].tolist(),
                    payload={
                        "candidate_id":      str(row["candidate_id"]),
                        "composite_score":   float(row.get("composite_score",  0.0)),
                        "behavioral_score":  float(row.get("behavioral_score", 0.5)),
                        "extraction_score":  float(row.get("extraction_score", 0.3)),
                        "honeypot_risk":     float(row.get("honeypot_risk",    0.0)),
                        "red_flags":         str(row.get("red_flags", "")),
                        "notice_days":       int(row.get("notice_days", 60)),
                        "loc_multiplier":    float(row.get("loc_multiplier",    1.0)),
                        "notice_multiplier": float(row.get("notice_multiplier", 1.0)),
                        "blob_career":       str(row.get("blob_career", "")),
                        "blob_skills":       str(row.get("blob_skills", "")),
                        "implicit_evidence":     "",
                        "key_concern":           None,
                        "top_skill_signal":      "",
                        "deployment_confirmed":  False,
                    }
                ) for j, (_, row) in enumerate(chunk_df.iterrows())
            ]
            client.upsert(collection_name=f"candidates_{aspect}", points=points)
        del chunk_df; gc.collect()

    df_1b = pd.read_csv(PHASE1B_CSV)
    for _, row in df_1b.iterrows():
        pid = cid_to_point_id(row["candidate_id"])
        patch = {
            "composite_score":      float(row.get("enriched_composite_score", row.get("composite_score", 0.0))),
            "implicit_evidence":    str(row.get("implicit_evidence", "")),
            "key_concern":          row.get("key_concern"),
            "top_skill_signal":     str(row.get("top_skill_signal", "")),
            "deployment_confirmed": bool(row.get("deployment_confirmed_1b", False)),
        }
        for aspect in ASPECT_WEIGHTS:
            try:
                client.set_payload(
                    collection_name=f"candidates_{aspect}",
                    payload=patch,
                    points=PointIdsList(points=[pid])
                )
            except Exception:
                pass

    del bge; flush()
    return client

# ══════════════════════════════════════════════════════════
# ENTRYPOINT
# ══════════════════════════════════════════════════════════
if __name__ == "__main__":
    import docx

    setup_local_models()

    doc = docx.Document(JD_PATH)
    jd_content = "\n".join(
        p.text for p in doc.paragraphs if p.text.strip()
    )

    jd_context_path = os.path.join(DRIVE_DIR, "jd_context.json")

    if os.path.exists(jd_context_path):
        with open(jd_context_path, "r") as f:
            jd_context = json.load(f)
        print("✅ Loaded jd_context from cache")
    else:
        jd_context = run_phase0_jd_parsing(jd_content)
        with open(jd_context_path, "w") as f:
            json.dump(jd_context, f)
        print("✅ Saved jd_context to cache")

    # Phase 1A
    run_phase1a_mass_extraction(DATA_PATH, jd_context)

    # Phase 1B
    run_phase1b_evidence_reasoning(jd_context)

    # Phase 1C
    qdrant_client = run_phase1c_full_vector_ingest()

    print("\n🎉 Phase 1 Pipeline Complete! Ready for Phase 2 decoupling.")

⚠️ RUNNING IN TEST MODE (200 Candidates)
📥 CHECKING LOCAL MODEL STORAGE
✅ Models ready.

✅ Loaded jd_context from cache
⚡ PHASE 1A: MASS EXTRACTION
WARNING 06-30 07:25:15 config.py:306] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 06-30 07:25:15 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=3072, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda,

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 06-30 07:25:41 model_runner.py:1071] Loading model weights took 5.1959 GB
INFO 06-30 07:25:44 gpu_executor.py:122] # GPU blocks: 4933, # CPU blocks: 1170
INFO 06-30 07:25:44 gpu_executor.py:126] Maximum concurrency for 3072 tokens per request: 25.69x
INFO 06-30 07:25:44 model_runner.py:1402] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-30 07:25:44 model_runner.py:1406] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-30 07:25:57 model_runner.py:1530] Graph capturing finished in 13 secs.


1A chunks: 100%|██████████| 1/1 [01:42<00:00, 102.62s/it]


✅ Phase 1A complete → /content/drive/MyDrive/redrob_ai/TEST_v13_phase1a.csv

🧠 PHASE 1B: EVIDENCE REASONING
  Strata pool selected: 200 candidates
WARNING 06-30 07:27:43 config.py:306] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
INFO 06-30 07:27:43 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=4096, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=False, kv_cache_dtype=auto, quantization_param_path=None, device_config=cuda, 

...

INFO 06-30 07:28:08 model_runner.py:1071] Loading model weights took 5.1959 GB
INFO 06-30 07:28:11 gpu_executor.py:122] # GPU blocks: 4800, # CPU blocks: 1170
INFO 06-30 07:28:11 gpu_executor.py:126] Maximum concurrency for 4096 tokens per request: 18.75x
INFO 06-30 07:28:11 model_runner.py:1402] Capturing the model for CUDA graphs. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.
INFO 06-30 07:28:11 model_runner.py:1406] CUDA graphs can take additional 1~3 GiB memory per GPU. If you are running out of memory, consider decreasing `gpu_memory_utilization` or enforcing eager mode. You can also reduce the `max_num_seqs` as needed to decrease memory usage.
INFO 06-30 07:28:16 model_runner.py:1530] Graph capturing finished in 5 secs.


Processed prompts: 100%|██████████| 200/200 [02:49<00:00,  1.18it/s, est. speed input: 715.92 toks/s, output: 106.90 toks/s]


✅ Phase 1B complete → /content/drive/MyDrive/redrob_ai/TEST_v8.2.2_phase1b.csv

🧬 PHASE 1C: BGE-M3 ENCODING


100%|██████████| 1/1 [00:27<00:00, 27.64s/it]



🎉 Phase 1 Pipeline Complete! Ready for Phase 2 decoupling.


Retrieve Top 150 (Pipeline A)



In [1]:
import os, gc, pandas as pd, torch
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient

# CRITICAL: Match the IS_TEST_RUN state from phase1_pipeline
IS_TEST_RUN = True

DRIVE_DIR = "/content/drive/MyDrive/redrob_ai"
QDRANT_PATH = os.path.join(DRIVE_DIR, "TEST_v58_qdrant") if IS_TEST_RUN else "/content/qdrant_storage"
TOP150_CSV = os.path.join(DRIVE_DIR, "top150_candidates.csv")

def run(client, jd_text):
    bge_q = SentenceTransformer(os.path.join(DRIVE_DIR, "models", "bge-m3"))
    jd_vec = bge_q.encode([jd_text[:1500]])[0].tolist()
    del bge_q; gc.collect(); torch.cuda.empty_cache()

    merged = {}
    for aspect, weight in {"skills": 0.50, "career": 0.40, "education": 0.10}.items():
        for h in client.query_points(collection_name=f"candidates_{aspect}", query=jd_vec, limit=300).points:
            cid = h.payload.get("candidate_id")
            if cid not in merged: merged[cid] = {"payload": h.payload, "semantic_score": 0.0}
            merged[cid]["semantic_score"] += h.score * weight

    cands = sorted([v for v in merged.values() if float(v["payload"].get("honeypot_risk", 0)) < 0.80], key=lambda x: x["semantic_score"], reverse=True)[:150]
    df = pd.DataFrame([{**c["payload"], "semantic_score": c["semantic_score"]} for c in cands])
    df.to_csv(TOP150_CSV, index=False)
    print(f"✅ Retrieved top 150 candidates semantically.")

if __name__ == "__main__":
    import docx
    run(QdrantClient(path=QDRANT_PATH), "\n".join(p.text for p in docx.Document(os.path.join(DRIVE_DIR, "job_description.docx")).paragraphs if p.text.strip()))

✅ Retrieved top 150 candidates semantically.


RERANKER (CPU INFERENCE AND RUNS UNDER 5 MINS(CONSIDERING THE EXCLUSION OF DEPENDENCIES INSTALL TIME))

In [2]:
import os, gc, pandas as pd, numpy as np, torch
from sentence_transformers import CrossEncoder

DRIVE_DIR = "/content/drive/MyDrive/redrob_ai"
TOP150_CSV = os.path.join(DRIVE_DIR, "top150_candidates.csv")
RERANKER_CSV = os.path.join(DRIVE_DIR, "reranker_output.csv")

def run(jd_text):
    try: df = pd.read_csv(TOP150_CSV).fillna("")
    except: return
    pairs = [(jd_text[:512], (str(r.get("blob_skills", "")) + " " + str(r.get("blob_career", "")))[:512]) for _, r in df.iterrows()]
    reranker_scores, loaded = np.full(len(df), 0.5), False
    for model in ["BAAI/bge-reranker-base", "cross-encoder/ms-marco-MiniLM-L-6-v2"]:
        try:
            rx = CrossEncoder(model, max_length=512, device="cuda" if torch.cuda.is_available() else "cpu")
            arr = np.array(rx.predict(pairs, batch_size=32, show_progress_bar=False), dtype=float)
            reranker_scores = (arr - arr.min()) / (arr.max() - arr.min()) if arr.max() > arr.min() else np.full(len(df), 0.5)
            del rx; gc.collect(); torch.cuda.empty_cache()
            loaded = True; break
        except: pass
    pd.DataFrame({"candidate_id": df["candidate_id"], "reranker_score": reranker_scores, "reranker_loaded": loaded}).to_csv(RERANKER_CSV, index=False)
    print(f"✅ CrossEncoder reranking complete.")

if __name__ == "__main__":
    import docx
    run("\n".join(p.text for p in docx.Document(os.path.join(DRIVE_DIR, "job_description.docx")).paragraphs if p.text.strip()))

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

✅ CrossEncoder reranking complete.


Ensemble & Submission (Pipeline C)

In [3]:
import os, sys, gc, pandas as pd, torch

# 🛑 CRITICAL FIX FOR COLAB
if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: 1
if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2
os.environ["VLLM_USE_V1"] = "0"

from vllm import LLM, SamplingParams

IS_TEST_RUN = True
DRIVE_DIR = "/content/drive/MyDrive/redrob_ai"
TOP150_CSV, RERANKER_CSV = os.path.join(DRIVE_DIR, "top150_candidates.csv"), os.path.join(DRIVE_DIR, "reranker_output.csv")
FINAL_CSV = os.path.join(DRIVE_DIR, "TEST_v8_submission.csv") if IS_TEST_RUN else os.path.join(DRIVE_DIR, "team_cuda,coffee,code_submission004.csv")

def run():
    df = pd.merge(pd.read_csv(TOP150_CSV).fillna(""), pd.read_csv(RERANKER_CSV).fillna(""), on="candidate_id", how="inner")
    final = []
    for _, r in df.iterrows():
        sem, rer, comp, beh, ext = map(lambda k: float(r.get(k, 0.0) or 0.0) if k != "reranker_score" else float(r.get(k, 0.5) or 0.5), ["semantic_score", "reranker_score", "composite_score", "behavioral_score", "extraction_score"])
        raw = (0.25 * sem + 0.35 * rer + 0.20 * comp + 0.12 * beh + 0.08 * ext) if str(r.get("reranker_loaded", "False")).lower() in ("true", "1", "yes") else (0.25 * sem + 0.35 * comp + 0.10 * beh + 0.30 * ext)
        final.append({**r.to_dict(), "ensemble_score": round(max(0.0, min(raw + (float(r.get("loc_multiplier", 1.0) or 1.0) - 0.80) * 0.10 + (float(r.get("notice_multiplier", 1.0) or 1.0) - 0.80) * 0.05, 1.0)), 4)})

    top_100 = sorted(final, key=lambda x: x["ensemble_score"], reverse=True)[:100]

    nat = LLM(model=os.path.join(DRIVE_DIR, "models", "Qwen2.5-7B-Instruct-AWQ"), quantization="awq", max_model_len=768, gpu_memory_utilization=0.70, dtype="float16", enforce_eager=True)
    prompts = [f"<|im_start|>system\nWrite 2 sentences. 1: strongest signal. 2: main risk.\n<|im_end|>\n<|im_start|>user\nPositive: {(str(c.get('top_skill_signal','')).strip() or str(c.get('implicit_evidence','')).strip() or str(c.get('blob_skills',''))[:200] or 'ML background')[:300]}\nConcern: {(str(c.get('key_concern') or '').strip() or str(c.get('red_flags','none')).strip(' |') or 'none')[:200]}\n<|im_end|>\n<|im_start|>assistant\n" for c in top_100]
    cots = [o.outputs[0].text.strip() for o in nat.generate(prompts, SamplingParams(temperature=0.25, max_tokens=120))]

    rows = []
    for rank, (cand, cot) in enumerate(zip(top_100, cots), 1):
        f = " | ".join(filter(bool, [(str(cand.get("red_flags", "")).strip(" |") if str(cand.get("red_flags", "")).lower() not in ("nan", "none") else "").replace("no_prod_deployment", "") if str(cand.get("deployment_confirmed", "False")).lower() in ("true", "1", "yes") else (str(cand.get("red_flags", "")).strip(" |") if str(cand.get("red_flags", "")).lower() not in ("nan", "none") else ""), str(cand.get("key_concern", "")).strip() if str(cand.get("key_concern", "")).lower() not in ("nan", "none", "null") else ""]))
        s = cand["ensemble_score"]
        rows.append({"rank": rank, "candidate_id": cand["candidate_id"], "score": s, "tier": "T1" if s >= 0.6 else "T2" if s >= 0.5 else "T3" if s >= 0.4 else "T4" if s >= 0.3 else "T5", "reasoning": cot, "red_flags": f or "None"})
    pd.DataFrame(rows).to_csv(FINAL_CSV, index=False)
    print(f"✅ Final CSV Generated!")

if __name__ == "__main__": run()

/usr/local/lib/python3.12/dist-packages/vllm/connections.py:8: RuntimeWarning: Failed to read commit hash:
No module named 'vllm._version'
  from vllm.version import __version__ as VLLM_VERSION


WARNING 06-30 07:35:38 config.py:306] awq quantization is not fully optimized yet. The speed can be slower than non-quantized models.
WARNING 06-30 07:35:38 config.py:380] To see benefits of async output processing, enable CUDA graph. Since, enforce-eager is enabled, async output processor cannot be used
INFO 06-30 07:35:38 llm_engine.py:237] Initializing an LLM engine (vdev) with config: model='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', speculative_config=None, tokenizer='/content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, rope_scaling=None, rope_theta=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=awq, enforce_eager=True, kv_cache_dtype=auto, quantization_param_path=Non

/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:211: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_fwd")
/usr/local/lib/python3.12/dist-packages/xformers/ops/fmha/flash.py:344: FutureWarning: `torch.library.impl_abstract` was renamed to `torch.library.register_fake`. Please use that instead; we will remove `torch.library.impl_abstract` in a future version of PyTorch.
  @torch.library.impl_abstract("xformers_flash::flash_bwd")


INFO 06-30 07:35:41 model_runner.py:1060] Starting to load model /content/drive/MyDrive/redrob_ai/models/Qwen2.5-7B-Instruct-AWQ...
INFO 06-30 07:35:41 selector.py:224] Cannot use FlashAttention-2 backend for Volta and Turing GPUs.
INFO 06-30 07:35:41 selector.py:115] Using XFormers backend.


Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


INFO 06-30 07:36:06 model_runner.py:1071] Loading model weights took 5.2035 GB
INFO 06-30 07:36:08 gpu_executor.py:122] # GPU blocks: 2803, # CPU blocks: 4681
INFO 06-30 07:36:08 gpu_executor.py:126] Maximum concurrency for 768 tokens per request: 58.40x


Processed prompts: 100%|██████████| 100/100 [00:19<00:00,  5.06it/s, est. speed input: 252.32 toks/s, output: 245.40 toks/s]


✅ Final CSV Generated!


In [31]:
# 1. Clean the environment (prevents ghost db errors)
!rm -rf /content/drive/MyDrive/redrob_ai/TEST_v8_qdrant
!rm -f /content/drive/MyDrive/redrob_ai/TEST_v13_phase1a.csv

# 2. Re-run Phase 1 (Extract, Reason, Ingest)
!python phase1_pipeline.py

# 3. Retrieve Top 150
!python retrieve_top150.py

# 4. Rerank
!python reranker.py

# 5. Build submission
!python submission.py

python3: can't open file '/content/phase1_pipeline.py': [Errno 2] No such file or directory
🎯 PHASE 2A: RETRIEVE TOP 150
Traceback (most recent call last):
  File "/content/retrieve_top150.py", line 72, in <module>
    run(q_client, jd_content)
  File "/content/retrieve_top150.py", line 38, in run
    hits = client.query_points(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/qdrant_client/qdrant_client.py", line 423, in query_points
    return self._client.query_points(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/qdrant_client/local/qdrant_local.py", line 407, in query_points
    collection = self._get_collection(collection_name)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/qdrant_client/local/qdrant_local.py", line 183, in _get_collection
    raise ValueError(f"Collection {collection_name} not found")
ValueError: Collection candidates_skills not found
Ex